# Trabajo Práctico 3: Redes neuronales

Cada sección tiene un responsable (ver `JOURNAL.md`). Agregar celdas solo debajo
de los encabezados propios y no mover ni editar celdas de otras secciones.

## 0. Configuración común

*Acordada en T0 entre los tres. Los cambios posteriores se registran en `JOURNAL.md`
y se avisan al equipo.*

- **Rutas.** Todas se construyen desde la raíz del repositorio, por lo que el notebook
  funciona igual si el kernel arranca en la raíz o en `notebooks/`. Los CSV de
  UNSW-NB15 van en `data/raw/unsw_nb15/` y quedan fuera de Git.
- **Semillas.** Cada sección crea su propio generador con su semilla, por ejemplo
  `rng_s2 = np.random.default_rng(SEEDS["s2_xor"])`, y usa nombres de variables con su
  prefijo. Así sus resultados no dependen de lo que se ejecutó antes.
- **Ejecución reducida.** Con la variable de entorno `TP3_QUICK=1`, las secciones pueden
  reducir épocas o repeticiones con `quick_or_full(reducido, completo)` mientras se
  desarrolla. La ejecución final (T8) se hace sin esa variable, con los valores del
  enunciado.
- **Normalización L2.** `USE_L2` queda en `None` hasta que §4.1 (T4) decida; §4.2 y §4.4
  reciben la bandera como parámetro.
- **Pruebas.** Cada sección ejecuta solo sus propias clases de prueba con un
  `unittest.TestSuite` explícito, como en §4.3, y usa nombres de clases que no se
  repitan entre secciones.

In [ ]:
import os
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn


def find_project_root(start):
    """Busca hacia arriba desde `start` la carpeta que contiene `JOURNAL.md` y `notebooks/`."""
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "JOURNAL.md").is_file() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError(f"No se encontró la raíz del proyecto desde {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
UNSW_DIR = DATA_RAW_DIR / "unsw_nb15"
RESULTS_FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
RESULTS_TABLES_DIR = PROJECT_ROOT / "results" / "tables"

# Una semilla por sección; cada sección crea su propio generador a partir de la suya.
SEEDS = {
    "s1_perceptron": 1100,
    "s2_xor": 2100,
    "s3_r2": 3100,
    "s4_datos": 4000,
    "s4_1_l2": 4100,
    "s4_2_ensamble": 4200,
    "s4_3_ece": 4300,
    "s4_4_evaluacion": 4400,
}

# Modo de ejecución reducida para el desarrollo (TP3_QUICK=1). T8 se ejecuta sin él.
QUICK = os.environ.get("TP3_QUICK", "0") == "1"


def quick_or_full(quick_value, full_value):
    """Devuelve `quick_value` en ejecución reducida y `full_value` en la ejecución completa."""
    return quick_value if QUICK else full_value


# Decisión de §4.1 (T4): True o False. Mientras sea None, §4.2 y §4.4 la reciben como parámetro.
USE_L2 = None

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Python {platform.python_version()} en {platform.platform()} ({platform.machine()})")
print(
    f"numpy {np.__version__} · pandas {pd.__version__} · scipy {scipy.__version__} · "
    f"scikit-learn {sklearn.__version__} · matplotlib {matplotlib.__version__}"
)
print(f"Ejecución reducida (TP3_QUICK): {QUICK}")
if sys.version_info[:3] != (3, 15, 0):
    print("Aviso: la versión acordada en T0 es Python 3.15.0 (ver requirements.txt).")

## 1. Perceptrón de tres capas

*Responsable: P1 (T1).*

### 1.1 Clase `Multilayer_perceptron`

La red tiene tres capas: $D$ neuronas de entrada, $M$ ocultas con tangente hiperbólica y $K$ de salida
con softmax. Sigue el contrato de la red acordado en T0 (`JOURNAL.md`), que también usan §3 y §4.1:

| Método | Qué hace |
| --- | --- |
| `__init__(neurons_per_layer, alpha, max_weights, gamma=0.0, seed=None)` | Guarda $[D, M, K]$, $\alpha$, $\gamma$ y `max_weights`; inicializa los pesos de forma uniforme en $[-\texttt{max\_weights}, \texttt{max\_weights}]$ con un generador propio creado con `seed`. Agrega `gamma`, que la lista de argumentos del enunciado pide aunque la firma impresa lo omite (D2). |
| `forward(X)` | Propagación hacia adelante de la matriz $X$ ($n \times D$, una observación por fila); devuelve la salida $n \times K$ y guarda los pesos netos y las salidas de cada capa. |
| `backpropagate_deltas(X, T)` | Calcula los deltas de las dos capas y, a partir de ellos, el gradiente del error de `get_error` respecto de cada matriz de pesos. |
| `update_weights(X, T)` | Actualiza ambas matrices con AdaGrad y momentum (D5) y recorta los pesos a $\pm$`max_weights`. |
| `train_mlp(num_epochs, X, T, alpha, gamma, X_val=None, T_val=None)` | Repite `update_weights` en lote completo y registra el error de entrenamiento (y de validación, si se pasa) después de cada época, para graficarlo. |
| `get_error(X, T)` | MSE: media de $(y - t)^2$ sobre las $n$ observaciones y las $K$ salidas. |
| `predict(X)` | Etiquetas enteras: argmax de la salida si $K \ge 2$; umbral 0.5 si $K = 1$. |

**Representación matricial.** Como en el material del curso, el sesgo es la fila 0 de cada matriz de
pesos y se agrega una entrada constante igual a 1 a cada capa: $W_o$ es $(D+1) \times M$ y $W_s$ es
$(M+1) \times K$. `T` es una matriz $n \times K$ de ceros y unos: one-hot si $K \ge 2$ y una columna con
1 = clase positiva si $K = 1$.

**Salida con $K = 1$ (D3).** Una softmax de una sola componente vale siempre 1 y su derivada es 0, de
modo que la red no aprendería. Con $K = 1$ la salida es la primera componente de
$\operatorname{softmax}([p, 0])$, que sigue siendo una softmax y es igual a la sigmoide:

$$\operatorname{softmax}([p, 0])_1 = \frac{e^{p}}{e^{p} + e^{0}} = \frac{1}{1 + e^{-p}} = \sigma(p).$$

El docente indicó que una salida igual a 1 con $K = 1$ también era aceptable; se eligió esta forma
porque permite entrenar y comparar las configuraciones de §3.

### 1.2 Propagación, deltas y AdaGrad con momentum

**Propagación hacia adelante.** Con $\tilde X = [\mathbf 1, X]$ ($n \times (D+1)$):

$$A_o = \tilde X W_o, \qquad Y_o = \tanh(A_o), \qquad \tilde Y_o = [\mathbf 1, Y_o], \qquad
A_s = \tilde Y_o W_s, \qquad Y_s = \operatorname{softmax}(A_s),$$

donde la softmax se aplica por fila: $y_{ik} = e^{a_{ik}} / \sum_j e^{a_{ij}}$. Antes de exponenciar se
resta el máximo de cada fila, lo que no cambia el resultado y evita desbordamientos.

**Error.** `get_error` es el MSE promediado sobre las $n$ observaciones y las $K$ salidas:

$$E = \frac{1}{nK} \sum_{i=1}^{n} \sum_{k=1}^{K} (y_{ik} - t_{ik})^2, \qquad
e_{ik} = \frac{\partial E}{\partial y_{ik}} = \frac{2}{nK}\,(y_{ik} - t_{ik}).$$

Los deltas de la clase calculan el gradiente exacto de este $E$, con el promedio y sin el factor
$\tfrac12$ que suele agregarse para simplificar la derivada. Así la prueba de gradiente numérico compara los deltas
directamente con `get_error`. AdaGrad divide cada paso entre la raíz de los gradientes acumulados, por
lo que un factor constante en el error casi no cambia el entrenamiento.

**Delta de salida con softmax y MSE.** Definimos $\delta^s_{ij} = \partial E / \partial a^s_{ij}$. En
una softmax, cada salida depende de todos los netos de su fila; el jacobiano es

$$\frac{\partial y_{ik}}{\partial a_{ij}} = y_{ik}\,(\mathbb 1[k = j] - y_{ij}),$$

y por la regla de la cadena

$$\delta^s_{ij} = \sum_{k=1}^{K} e_{ik}\, y_{ik}\,(\mathbb 1[k = j] - y_{ij})
= y_{ij}\Big(e_{ij} - \sum_{k=1}^{K} e_{ik}\, y_{ik}\Big).$$

Como $\sum_j y_{ij} = 1$, cada fila de $\delta^s$ suma cero; una de las pruebas lo verifica.

**Caso $K = 1$.** Con $y_i = \operatorname{softmax}([p_i, 0])_1 = \sigma(p_i)$, solo la primera
componente entra en el error. La fórmula anterior con $e = [e_i, 0]$ e $y = [y_i, 1 - y_i]$ da

$$\delta^s_i = y_i\,(e_i - e_i\, y_i) = e_i\, y_i\,(1 - y_i),$$

que tiene la forma del delta con sigmoide del material del curso.

**Delta de la capa oculta.** Se retropropaga por $W_s$ sin la fila del sesgo (la neurona constante no
depende de la capa anterior), y la derivada de la tangente hiperbólica es $1 - \tanh^2$:

$$\delta^o_{im} = \big(1 - (y^o_{im})^2\big) \sum_{k=1}^{K} \delta^s_{ik}\,(W_s)_{mk}, \qquad
\delta^o = (1 - Y_o \odot Y_o) \odot \big(\delta^s\, W_s[1{:}, :]^\top\big).$$

**Gradientes.** Como los deltas ya incluyen el factor $1/(nK)$, el gradiente del error medio es la
suma sobre las observaciones:

$$\nabla_{W_s} E = \tilde Y_o^\top\, \delta^s, \qquad \nabla_{W_o} E = \tilde X^\top\, \delta^o.$$

**AdaGrad con momentum (D5).** En cada época, con todo el lote y para cada matriz de pesos $W$ con
gradiente $g$:

$$G \leftarrow G + g \odot g, \qquad v \leftarrow \gamma\, v + \frac{\alpha\, g}{\sqrt{G} + \varepsilon},
\qquad W \leftarrow W - v,$$

y después $W$ se recorta a $[-\texttt{max\_weights}, \texttt{max\_weights}]$. Con $\gamma = 0$ queda AdaGrad
puro (Duchi et al., 2011). $G$ y $v$ empiezan en cero y se conservan entre llamadas a `train_mlp`. Se
usa $\varepsilon = 10^{-8}$, que solo evita dividir entre cero mientras la suma $G$ de un peso es 0 y es
despreciable frente a $\sqrt G$ en cuanto hay gradiente. Es del mismo orden que el $\delta \approx 10^{-7}$
de Goodfellow et al. (2016, algoritmo 8.4). En el primer paso $G = g^2$, de modo que cada peso se
mueve casi exactamente $\alpha \cdot \operatorname{sign}(g)$; una de las pruebas lo usa.

**Qué cambia respecto de la retropropagación vista en clase**

1. *Salida softmax.* El delta de salida usa el jacobiano completo de la softmax, porque cada salida
   depende de todos los netos de su fila. La derivada elemento a elemento $y(1 - y)$ de la sigmoide
   solo reaparece con $K = 1$.
2. *Capa oculta con tangente hiperbólica.* Su derivada es $1 - y^2$ en lugar de $y(1 - y)$.
3. *Error MSE promediado en lote completo.* El gradiente acumula las $n$ observaciones con el factor
   $2/(nK)$ y los pesos se actualizan una vez por época.
4. *Descenso del gradiente adaptativo.* El paso fijo $W \leftarrow W - \alpha \nabla E$ se reemplaza por
   un paso propio de cada peso, $\alpha / (\sqrt{G} + \varepsilon)$, que disminuye en los pesos con
   gradientes históricos grandes. El momentum $\gamma$ suma una fracción del paso anterior.
5. *Recorte.* Después de cada actualización los pesos se limitan a $\pm$`max_weights`.

**Referencias**

- Duchi, J., Hazan, E. y Singer, Y. (2011). Adaptive subgradient methods for online learning and
  stochastic optimization. *Journal of Machine Learning Research*, 12, 2121–2159.
- Goodfellow, I., Bengio, Y. y Courville, A. (2016). *Deep Learning*. MIT Press, sección 8.5.1.

In [ ]:
import numpy as np


class Multilayer_perceptron:
    """Perceptrón de tres capas (entrada, oculta y salida) entrenado con AdaGrad y momentum.

    Arquitectura `[D, M, K]`: D entradas, M neuronas ocultas con tangente hiperbólica
    y K salidas con softmax. Con K = 1 la salida es la primera componente de
    softmax([p, 0]), que equivale a sigmoide(p) (D3 en `JOURNAL.md`).

    Pesos en la representación matricial del curso, con el sesgo en la fila 0:
    `W_o` de forma (D + 1) × M (entrada → oculta) y `W_s` de forma (M + 1) × K
    (oculta → salida).

    Atributos que actualizan los métodos (valen None hasta la primera llamada):

    - `forward`: `X_b` (entrada con la columna de unos), `A_o` y `Y_o` (pesos netos
      y salidas de la capa oculta), `Y_o_b` (salidas ocultas con la columna de unos),
      `A_s` y `Y_s` (pesos netos y salidas de la capa de salida). Reflejan la última
      llamada, incluida la que hace `get_error`.
    - `backpropagate_deltas`: `delta_o` (n × M), `delta_s` (n × K) y los gradientes
      `grad_W_o` y `grad_W_s`, con la forma de sus pesos.
    - AdaGrad y momentum: `G_o`, `G_s` (suma de gradientes al cuadrado) y `v_o`,
      `v_s` (velocidad). Se conservan entre llamadas a `train_mlp`.
    """

    # ε de AdaGrad: evita dividir entre cero mientras la suma G de un peso es 0 (T1a).
    EPSILON = 1e-8

    def __init__(self, neurons_per_layer, alpha, max_weights, gamma=0.0, seed=None):
        """Crea la red e inicializa los pesos de forma uniforme en [−max_weights, max_weights].

        Parámetros
        ----------
        neurons_per_layer : secuencia de tres enteros positivos [D, M, K]
            Neuronas de entrada, de la capa oculta y de salida.
        alpha : float > 0
            Coeficiente de aprendizaje α.
        max_weights : float > 0
            Valor absoluto máximo de cualquier peso; se usa para inicializar y para
            recortar los pesos después de cada actualización.
        gamma : float en [0, 1)
            Coeficiente de momentum γ. Con 0 el entrenamiento es AdaGrad puro.
        seed : int o None
            Semilla del generador propio de la instancia (pesos iniciales).
        """
        capas = [int(n) for n in neurons_per_layer]
        if len(capas) != 3 or any(n != m for n, m in zip(capas, neurons_per_layer)) or min(capas) < 1:
            raise ValueError("neurons_per_layer debe ser [D, M, K] con tres enteros positivos")
        if not max_weights > 0:
            raise ValueError("max_weights debe ser positivo")
        self.neurons_per_layer = capas
        self.D, self.M, self.K = capas
        self.alpha, self.gamma = self._validar_alpha_gamma(alpha, gamma)
        self.max_weights = float(max_weights)
        self.rng = np.random.default_rng(seed)
        self.W_o = self.rng.uniform(-self.max_weights, self.max_weights, (self.D + 1, self.M))
        self.W_s = self.rng.uniform(-self.max_weights, self.max_weights, (self.M + 1, self.K))
        self.G_o, self.G_s = np.zeros_like(self.W_o), np.zeros_like(self.W_s)
        self.v_o, self.v_s = np.zeros_like(self.W_o), np.zeros_like(self.W_s)
        self.X_b = self.A_o = self.Y_o = self.Y_o_b = self.A_s = self.Y_s = None
        self.delta_o = self.delta_s = self.grad_W_o = self.grad_W_s = None

    @staticmethod
    def _validar_alpha_gamma(alpha, gamma):
        if not alpha > 0:
            raise ValueError("alpha debe ser positivo")
        if not 0 <= gamma < 1:
            raise ValueError("gamma debe estar en [0, 1)")
        return float(alpha), float(gamma)

    @staticmethod
    def _softmax(A):
        """Softmax por fila, estable numéricamente (resta el máximo de cada fila)."""
        E = np.exp(A - A.max(axis=1, keepdims=True))
        return E / E.sum(axis=1, keepdims=True)

    def _validar_X(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.D:
            raise ValueError(f"X debe tener forma n × {self.D}; se recibió {X.shape}")
        return X

    def _validar_T(self, T, n):
        T = np.asarray(T, dtype=float)
        if T.shape != (n, self.K):
            raise ValueError(f"T debe tener forma {(n, self.K)}; se recibió {T.shape}")
        if not np.all((T == 0) | (T == 1)):
            raise ValueError("T debe contener solo 0 y 1")
        if self.K >= 2 and not np.all(T.sum(axis=1) == 1):
            raise ValueError("Con K >= 2, cada fila de T debe ser one-hot")
        return T

    def forward(self, X):
        """Propagación hacia adelante; devuelve la salida n × K y guarda netos y salidas.

        A_o = [1, X] W_o, Y_o = tanh(A_o), A_s = [1, Y_o] W_s y Y_s = softmax(A_s)
        por fila. Con K = 1, Y_s es la primera componente de softmax([A_s, 0]).
        """
        X = self._validar_X(X)
        unos = np.ones((X.shape[0], 1))
        self.X_b = np.hstack([unos, X])
        self.A_o = self.X_b @ self.W_o
        self.Y_o = np.tanh(self.A_o)
        self.Y_o_b = np.hstack([unos, self.Y_o])
        self.A_s = self.Y_o_b @ self.W_s
        if self.K == 1:
            self.Y_s = self._softmax(np.hstack([self.A_s, np.zeros_like(self.A_s)]))[:, :1]
        else:
            self.Y_s = self._softmax(self.A_s)
        return self.Y_s

    def backpropagate_deltas(self, X, T):
        """Calcula los deltas de ambas capas y los gradientes del MSE de `get_error`.

        Los deltas son derivadas del error respecto de los pesos netos:
        delta_s = ∂E/∂A_s (n × K) y delta_o = ∂E/∂A_o (n × M). Ejecuta `forward(X)`.
        Devuelve `(delta_o, delta_s)` y guarda además `grad_W_o` y `grad_W_s`.
        """
        Y = self.forward(X)
        T = self._validar_T(T, Y.shape[0])
        n = Y.shape[0]
        e = 2.0 * (Y - T) / (n * self.K)                      # ∂E/∂Y_s
        if self.K == 1:
            self.delta_s = e * Y * (1.0 - Y)                  # softmax([p, 0]) = sigmoide(p)
        else:
            self.delta_s = Y * (e - np.sum(e * Y, axis=1, keepdims=True))  # jacobiano de softmax
        self.delta_o = (1.0 - self.Y_o ** 2) * (self.delta_s @ self.W_s[1:].T)
        self.grad_W_s = self.Y_o_b.T @ self.delta_s
        self.grad_W_o = self.X_b.T @ self.delta_o
        return self.delta_o, self.delta_s

    def _paso_adagrad(self, W, G, v, grad):
        """Aplica en el lugar un paso de AdaGrad con momentum y el recorte a max_weights."""
        G += grad * grad
        v *= self.gamma
        v += self.alpha * grad / (np.sqrt(G) + self.EPSILON)
        W -= v
        np.clip(W, -self.max_weights, self.max_weights, out=W)

    def update_weights(self, X, T):
        """Actualiza los pesos de ambas capas a partir de los deltas, con AdaGrad y momentum.

        Para cada matriz W con gradiente g: G ← G + g ⊙ g; v ← γ·v + α·g / (√G + ε);
        W ← W − v; y luego W se recorta a [−max_weights, max_weights]. Los gradientes
        se calculan con `backpropagate_deltas(X, T)` antes de modificar los pesos.
        """
        self.backpropagate_deltas(X, T)
        self._paso_adagrad(self.W_o, self.G_o, self.v_o, self.grad_W_o)
        self._paso_adagrad(self.W_s, self.G_s, self.v_s, self.grad_W_s)

    def get_error(self, X, T):
        """MSE: media de (y − t)² sobre las n observaciones y las K salidas."""
        Y = self.forward(X)
        T = self._validar_T(T, Y.shape[0])
        return float(np.mean((Y - T) ** 2))

    def train_mlp(self, num_epochs, X, T, alpha, gamma, X_val=None, T_val=None):
        """Entrena `num_epochs` épocas en lote completo y devuelve el error por época.

        `alpha` y `gamma` reemplazan los valores guardados en la instancia. G y v se
        conservan entre llamadas. Cada valor del historial se mide después de la
        actualización de su época. Devuelve
        `{"train_error": [...], "val_error": [...]}`; `val_error` queda vacía si no
        se pasan `X_val` y `T_val`.
        """
        if int(num_epochs) != num_epochs or num_epochs < 1:
            raise ValueError("num_epochs debe ser un entero positivo")
        if (X_val is None) != (T_val is None):
            raise ValueError("X_val y T_val se pasan juntos")
        self.alpha, self.gamma = self._validar_alpha_gamma(alpha, gamma)
        X = self._validar_X(X)
        T = self._validar_T(T, X.shape[0])
        historial = {"train_error": [], "val_error": []}
        for _ in range(int(num_epochs)):
            self.update_weights(X, T)
            historial["train_error"].append(self.get_error(X, T))
            if X_val is not None:
                historial["val_error"].append(self.get_error(X_val, T_val))
        return historial

    def predict(self, X):
        """Etiquetas enteras de forma (n,): argmax si K >= 2; umbral 0.5 (y >= 0.5 → 1) si K = 1."""
        Y = self.forward(X)
        if self.K == 1:
            return (Y[:, 0] >= 0.5).astype(int)
        return np.argmax(Y, axis=1)

### 1.3 Pruebas unitarias de §1

Las pruebas usan redes pequeñas, valores calculables a mano y datos generados con
`SEEDS["s1_perceptron"]`, y no dependen de otras secciones. La prueba central de los deltas compara
el gradiente que sale de `backpropagate_deltas` con un gradiente numérico de `get_error` por
diferencias centrales, $(E(w + h) - E(w - h)) / 2h$ con $h = 10^{-6}$, peso por peso.

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `__init__` | Red $[3, 5, 2]$ con `max_weights` = 0.5 | $W_o$ de $4 \times 5$, $W_s$ de $6 \times 2$, pesos en $[-0.5, 0.5]$, $G = v = 0$, $\gamma = 0$ por omisión |
| | Misma semilla y semilla distinta | Pesos iguales y distintos |
| | Capas, $\alpha$, `max_weights` o $\gamma$ inválidos | `ValueError` |
| `forward` | Red $[1, 1, 2]$ con pesos fijados y $x = 2$ | Netos, salidas y softmax calculados a mano |
| | Red $[4, 6, 3]$ con entradas grandes | Salida $10 \times 3$ finita, filas que suman 1 y formas de los atributos |
| | $K = 1$ | Salida igual a $\sigma(A_s)$ (D3) |
| | $X$ con columnas de más | `ValueError` |
| `backpropagate_deltas` | Gradiente numérico con $K = 3$ | Coincide con `grad_W_o` y `grad_W_s` (tolerancia relativa $10^{-5}$) |
| | Gradiente numérico con $K = 1$ | Coincide con `grad_W_o` y `grad_W_s` |
| | Formas de los deltas con softmax | $\delta^o$ de $n \times M$, $\delta^s$ de $n \times K$, filas de $\delta^s$ que suman 0 |
| `update_weights` | Primer paso con $\gamma = 0$ | $\Delta W = -\alpha g / (\lvert g\rvert + \varepsilon) \approx -\alpha \operatorname{sign}(g)$ |
| | Dos pasos con $\gamma = 0.9$ | $G = g_1^2 + g_2^2$ y $v_2 = \gamma v_1 + \alpha g_2 / (\sqrt{G} + \varepsilon)$ |
| | $\alpha = 5$ con `max_weights` = 0.3 | Ningún peso sale de $\pm 0.3$ y alguno queda en el borde |
| `get_error` | Pesos en cero (salida uniforme $1/K$) | 0.25 con $K = 1$ y $K = 2$; $(0.75^2 + 3 \cdot 0.25^2)/4$ con $K = 4$ |
| | Datos aleatorios con $K = 3$ | $\sum (y - t)^2 / (nK)$ |
| | `T` de forma $(n,)$ o filas no one-hot | `ValueError` |
| `train_mlp` | Dos nubes separables, 100 épocas, con validación | Historiales de 100 valores, error final < 10 % del inicial y validación bien clasificada |
| | 10 épocas seguidas frente a 4 + 6 épocas | Mismos pesos y errores ($G$ y $v$ se conservan); `val_error` vacía sin validación |
| `predict` | $K = 1$ con $p = 5 \tanh(x)$ | Etiquetas enteras $[0, 0, 1, 1]$ para $x = -2, -0.1, 0, 0.3$ |
| | $K = 3$ con pesos fijados | Argmax $[0, 1, 2]$ |

Antes de integrar T1a se comprobó que las pruebas fallan si se introducen errores a propósito: delta de
salida sin el jacobiano, derivada de la sigmoide en la capa oculta, otra escala del error, sin momentum,
sin acumular $G$, sin recorte, retropropagación por la fila del sesgo y umbral estricto en `predict`.

In [ ]:
import sys
import unittest


def s1_datos_prueba(n, D, K, rng):
    """Datos aleatorios de prueba: X normal n × D y T one-hot (K >= 2) o 0/1 (K = 1)."""
    X = rng.normal(size=(n, D))
    if K == 1:
        return X, rng.integers(0, 2, size=(n, 1)).astype(float)
    return X, np.eye(K)[rng.integers(0, K, size=n)]


def s1_gradiente_numerico(red, W, X, T, h=1e-6):
    """Gradiente de `get_error` respecto de la matriz W (atributo de `red`) por diferencias centrales."""
    grad = np.zeros_like(W)
    for idx in np.ndindex(W.shape):
        original = W[idx]
        W[idx] = original + h
        e_mas = red.get_error(X, T)
        W[idx] = original - h
        e_menos = red.get_error(X, T)
        W[idx] = original
        grad[idx] = (e_mas - e_menos) / (2 * h)
    return grad


class TestMLPConstructor(unittest.TestCase):
    def test_formas_rango_y_estado_inicial(self):
        red = Multilayer_perceptron([3, 5, 2], alpha=0.1, max_weights=0.5, seed=1)
        self.assertEqual(red.W_o.shape, (4, 5))
        self.assertEqual(red.W_s.shape, (6, 2))
        for W in (red.W_o, red.W_s):
            self.assertTrue(np.all(np.abs(W) <= 0.5))
        for M in (red.G_o, red.G_s, red.v_o, red.v_s):
            self.assertFalse(M.any())
        self.assertEqual((red.alpha, red.gamma, red.max_weights), (0.1, 0.0, 0.5))

    def test_semilla_reproducible(self):
        a = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=7)
        b = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=7)
        c = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=8)
        np.testing.assert_array_equal(a.W_o, b.W_o)
        np.testing.assert_array_equal(a.W_s, b.W_s)
        self.assertFalse(np.array_equal(a.W_o, c.W_o))

    def test_rechaza_argumentos_invalidos(self):
        for args in (([2, 3], 0.1, 1.0), ([2, 0, 1], 0.1, 1.0), ([2, 3, 1], 0.0, 1.0),
                     ([2, 3, 1], 0.1, -1.0)):
            with self.assertRaises(ValueError):
                Multilayer_perceptron(*args)
        with self.assertRaises(ValueError):
            Multilayer_perceptron([2, 3, 1], 0.1, 1.0, gamma=1.0)


class TestMLPForward(unittest.TestCase):
    def test_calculo_a_mano(self):
        red = Multilayer_perceptron([1, 1, 2], 0.1, 10.0)
        red.W_o = np.array([[0.5], [1.0]])            # a_o = 0.5 + x
        red.W_s = np.array([[0.0, 1.0], [2.0, 0.0]])  # a_s = [2·y_o, 1]
        y_o = np.tanh(0.5 + 2.0)
        z = np.exp([2 * y_o, 1.0])
        Y = red.forward(np.array([[2.0]]))
        np.testing.assert_allclose(red.A_o, [[2.5]])
        np.testing.assert_allclose(red.Y_o, [[y_o]])
        np.testing.assert_allclose(red.A_s, [[2 * y_o, 1.0]])
        np.testing.assert_allclose(Y, [z / z.sum()])

    def test_salida_softmax_y_formas(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([4, 6, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X = rng.normal(size=(10, 4)) * 50   # entradas grandes: softmax estable
        Y = red.forward(X)
        self.assertEqual(Y.shape, (10, 3))
        self.assertEqual((red.X_b.shape, red.Y_o_b.shape, red.A_s.shape), ((10, 5), (10, 7), (10, 3)))
        np.testing.assert_allclose(Y.sum(axis=1), np.ones(10))
        self.assertTrue(np.all((Y >= 0) & (Y <= 1)) and np.all(np.isfinite(Y)))

    def test_k1_equivale_a_sigmoide(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 1], 0.1, 2.0, seed=SEEDS["s1_perceptron"])
        Y = red.forward(rng.normal(size=(8, 2)))
        self.assertEqual(Y.shape, (8, 1))
        np.testing.assert_allclose(Y, 1.0 / (1.0 + np.exp(-red.A_s)))

    def test_rechaza_columnas_incorrectas(self):
        red = Multilayer_perceptron([2, 3, 1], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red.forward(np.zeros((4, 3)))


class TestMLPBackpropagateDeltas(unittest.TestCase):
    def _comprobar_gradiente(self, D, M, K):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([D, M, K], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(7, D, K, rng)
        red.backpropagate_deltas(X, T)
        grad_o, grad_s = red.grad_W_o.copy(), red.grad_W_s.copy()
        np.testing.assert_allclose(grad_o, s1_gradiente_numerico(red, red.W_o, X, T), rtol=1e-5, atol=1e-9)
        np.testing.assert_allclose(grad_s, s1_gradiente_numerico(red, red.W_s, X, T), rtol=1e-5, atol=1e-9)

    def test_gradiente_numerico_softmax(self):
        self._comprobar_gradiente(D=3, M=4, K=3)

    def test_gradiente_numerico_k1(self):
        self._comprobar_gradiente(D=2, M=5, K=1)

    def test_formas_y_suma_cero_de_delta_salida(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([3, 4, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(6, 3, 3, rng)
        delta_o, delta_s = red.backpropagate_deltas(X, T)
        self.assertEqual((delta_o.shape, delta_s.shape), ((6, 4), (6, 3)))
        # Con softmax, sumar la misma constante a los netos de una fila no cambia la salida:
        np.testing.assert_allclose(delta_s.sum(axis=1), np.zeros(6), atol=1e-15)


class TestMLPUpdateWeights(unittest.TestCase):
    def test_primer_paso_es_alpha_por_signo(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 2], alpha=0.05, max_weights=10.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(5, 2, 2, rng)
        W_o, W_s = red.W_o.copy(), red.W_s.copy()
        red.update_weights(X, T)
        for antes, despues, g in ((W_o, red.W_o, red.grad_W_o), (W_s, red.W_s, red.grad_W_s)):
            np.testing.assert_allclose(antes - despues, 0.05 * g / (np.abs(g) + Multilayer_perceptron.EPSILON))
            grande = np.abs(g) > 1e-4
            np.testing.assert_allclose((antes - despues)[grande], 0.05 * np.sign(g[grande]), rtol=1e-3)

    def test_momentum_y_acumulacion(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 1], alpha=0.1, max_weights=10.0, gamma=0.9, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(6, 2, 1, rng)
        red.update_weights(X, T)
        g1, v1 = red.grad_W_s.copy(), red.v_s.copy()
        W_antes = red.W_s.copy()
        red.update_weights(X, T)
        g2 = red.grad_W_s
        v2 = 0.9 * v1 + 0.1 * g2 / (np.sqrt(g1 ** 2 + g2 ** 2) + Multilayer_perceptron.EPSILON)
        np.testing.assert_allclose(red.G_s, g1 ** 2 + g2 ** 2)
        np.testing.assert_allclose(red.v_s, v2)
        np.testing.assert_allclose(W_antes - red.W_s, v2)

    def test_recorte_a_max_weights(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 2], alpha=5.0, max_weights=0.3, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(5, 2, 2, rng)
        red.update_weights(X, T)
        for W in (red.W_o, red.W_s):
            self.assertTrue(np.all(np.abs(W) <= 0.3))
        self.assertTrue(np.any(np.isclose(np.abs(red.W_s), 0.3)))


class TestMLPGetError(unittest.TestCase):
    def test_pesos_cero_dan_salida_uniforme(self):
        X = np.ones((4, 2))
        for K, T, esperado in ((2, np.eye(2)[[0, 1, 1, 0]], 0.25), (1, np.array([[0.], [1.], [1.], [0.]]), 0.25),
                               (4, np.eye(4), ((0.75 ** 2) + 3 * 0.25 ** 2) / 4)):
            red = Multilayer_perceptron([2, 3, K], 0.1, 1.0)
            red.W_o[:] = 0.0
            red.W_s[:] = 0.0
            self.assertAlmostEqual(red.get_error(X, T), esperado)

    def test_coincide_con_la_definicion(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([3, 4, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(9, 3, 3, rng)
        Y = red.forward(X)
        self.assertAlmostEqual(red.get_error(X, T), np.sum((Y - T) ** 2) / (9 * 3))

    def test_rechaza_t_con_forma_incorrecta(self):
        red = Multilayer_perceptron([2, 3, 1], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red.get_error(np.zeros((4, 2)), np.zeros(4))   # (n,) en lugar de (n, 1)
        red2 = Multilayer_perceptron([2, 3, 2], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red2.get_error(np.zeros((2, 2)), np.array([[1., 1.], [0., 1.]]))  # no es one-hot


class TestMLPTrain(unittest.TestCase):
    @staticmethod
    def _dos_nubes(n, rng):
        """Dos nubes gaussianas separables en R², con etiquetas one-hot."""
        X = np.vstack([rng.normal(-1.5, 0.6, size=(n, 2)), rng.normal(1.5, 0.6, size=(n, 2))])
        T = np.eye(2)[np.repeat([0, 1], n)]
        return X, T

    def test_aprende_problema_separable(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        X, T = self._dos_nubes(40, rng)
        X_val, T_val = self._dos_nubes(20, rng)
        red = Multilayer_perceptron([2, 4, 2], 0.05, 2.0, seed=SEEDS["s1_perceptron"])
        error_inicial = red.get_error(X, T)
        historial = red.train_mlp(100, X, T, alpha=0.05, gamma=0.5, X_val=X_val, T_val=T_val)
        self.assertEqual((len(historial["train_error"]), len(historial["val_error"])), (100, 100))
        self.assertLess(historial["train_error"][-1], 0.1 * error_inicial)
        self.assertAlmostEqual(historial["train_error"][-1], red.get_error(X, T))
        np.testing.assert_array_equal(red.predict(X_val), np.argmax(T_val, axis=1))

    def test_estado_se_conserva_entre_llamadas(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(10, 2, 1, rng)
        continua = Multilayer_perceptron([2, 3, 1], 0.1, 1.0, seed=3)
        por_partes = Multilayer_perceptron([2, 3, 1], 0.1, 1.0, seed=3)
        h = continua.train_mlp(10, X, T, alpha=0.02, gamma=0.8)
        h1 = por_partes.train_mlp(4, X, T, alpha=0.02, gamma=0.8)
        h2 = por_partes.train_mlp(6, X, T, alpha=0.02, gamma=0.8)
        np.testing.assert_allclose(continua.W_o, por_partes.W_o)
        np.testing.assert_allclose(continua.W_s, por_partes.W_s)
        np.testing.assert_allclose(h["train_error"], h1["train_error"] + h2["train_error"])
        self.assertEqual(h["val_error"], [])
        self.assertEqual((por_partes.alpha, por_partes.gamma), (0.02, 0.8))


class TestMLPPredict(unittest.TestCase):
    def test_umbral_k1(self):
        red = Multilayer_perceptron([1, 1, 1], 0.1, 10.0)
        red.W_o = np.array([[0.0], [1.0]])   # y_o = tanh(x)
        red.W_s = np.array([[0.0], [5.0]])   # p = 5·tanh(x)
        etiquetas = red.predict(np.array([[-2.0], [-0.1], [0.0], [0.3]]))
        np.testing.assert_array_equal(etiquetas, [0, 0, 1, 1])
        self.assertEqual(etiquetas.dtype.kind, "i")

    def test_argmax_k3(self):
        red = Multilayer_perceptron([2, 2, 3], 0.1, 10.0)
        red.W_o = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
        red.W_s = np.array([[0.0, 0.0, 0.5], [3.0, 0.0, 0.0], [0.0, 3.0, 0.0]])
        X = np.array([[2.0, 0.0], [0.0, 2.0], [0.0, 0.0]])
        np.testing.assert_array_equal(red.predict(X), [0, 1, 2])


suite_s1 = unittest.TestSuite()
for caso in (TestMLPConstructor, TestMLPForward, TestMLPBackpropagateDeltas, TestMLPUpdateWeights,
             TestMLPGetError, TestMLPTrain, TestMLPPredict):
    suite_s1.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s1 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s1)
assert resultado_s1.wasSuccessful(), "Fallaron pruebas unitarias de §1"

#### Captura de la prueba de entrenamiento

El enunciado (§1 d.1) pide un pantallazo de una prueba unitaria de `train_mlp`. Se toma en la ejecución
final del notebook (T8), con el código entregado: la salida de la celda anterior, con las líneas de
`TestMLPTrain`, se adjunta aquí como imagen de esta celda Markdown.

*Pendiente de T8.*

## 2. Clasificación de XOR

*Responsable: P1 (T1).*

## 3. Clasificador de datos en R²

*Responsable: P2 (T2a y T2b).*

### 3.2 b) Investigación: neuronas en la capa oculta frente a neuronas de entrada

*Pendiente de T2b:* la comparación de las gráficas, el ajuste de $\alpha$ según $M$ y la
relación entre $M$ y el error de entrenamiento se escriben después de ejecutar los
experimentos de esta sección.

**Pregunta.** ¿Puede ajustarse la cantidad de neuronas en la capa oculta $M$ en función
de las neuronas en la capa de entrada $D$?

**Lo que dice la literatura consultada.**

- **Existe una red suficiente, pero el teorema no dice de qué tamaño.** Hornik,
  Stinchcombe y White [1] demostraron que una red con una sola capa oculta puede
  aproximar cualquier función con la precisión deseada si dispone de suficientes
  neuronas ocultas. El resultado garantiza existencia; no da una fórmula para $M$.
- **Las cotas para memorizar los datos dependen de $N$, no de $D$.** Sartori y
  Antsaklis [2] muestran que una capa oculta con $p-1$ neuronas basta para ajustar
  exactamente $p$ patrones de entrenamiento. Huang [3] obtiene, para redes con dos
  capas ocultas, que unas $2\sqrt{(m+2)N}$ neuronas ocultas aprenden $N$ muestras
  distintas, con $m$ salidas. En ambos casos el tamaño queda determinado por el número
  de muestras y de salidas; la dimensión de entrada no aparece en la cota. Además,
  ajustar exactamente el entrenamiento no garantiza generalizar.
- **Cuando se busca generalizar, $D$ sí aparece.** Barron [4] acota el error cuadrático
  medio integrado de una red de una capa oculta con $n$ nodos, entrenada con $N$
  observaciones de dimensión $d$, por $O(C_f^2/n) + O\big(\tfrac{nd}{N}\log N\big)$,
  donde $C_f$ mide la complejidad de la función objetivo. El primer término
  (aproximación) baja al agregar neuronas; el segundo (estimación) crece con $n \cdot d$,
  que es del orden del número de pesos de la capa oculta. El balance se alcanza con
  $n \sim C_f \, \big(N/(d \log N)\big)^{1/2}$: con el mismo número de muestras, más
  entradas justifican **menos** neuronas ocultas, no más.
- **En la práctica no hay una regla exacta.** Sheela y Deepa [5] revisan unos 20 años
  de métodos para fijar el número de neuronas ocultas y comparan 101 criterios;
  advierten que una elección arbitraria produce sobreajuste o subajuste. Stathakis [6]
  concluye que no existe una solución exacta y busca la arquitectura con un algoritmo
  genético que favorece soluciones precisas y compactas.

**Conclusión.** Sí puede usarse $D$ para orientar la elección de $M$, pero no como única
variable: el tamaño adecuado depende también del número de muestras $N$, de las salidas
$K$ y de la complejidad de la frontera de decisión. Las reglas basadas solo en $D$ son
heurísticas de partida; $M$ debe calibrarse con un conjunto de validación.

**Referencias**

1. K. Hornik, M. Stinchcombe y H. White, “Multilayer feedforward networks are universal
   approximators”, *Neural Networks*, vol. 2, n.º 5, pp. 359–366, 1989.
   doi:10.1016/0893-6080(89)90020-8
2. M. A. Sartori y P. J. Antsaklis, “A simple method to derive bounds on the size and
   to train multilayer neural networks”, *IEEE Transactions on Neural Networks*,
   vol. 2, n.º 4, pp. 467–471, 1991. doi:10.1109/72.88168
3. G.-B. Huang, “Learning capability and storage capacity of two-hidden-layer
   feedforward networks”, *IEEE Transactions on Neural Networks*, vol. 14, n.º 2,
   pp. 274–281, 2003. doi:10.1109/TNN.2003.809401
4. A. R. Barron, “Approximation and estimation bounds for artificial neural networks”,
   *Machine Learning*, vol. 14, pp. 115–133, 1994. doi:10.1023/A:1022650905902
5. K. G. Sheela y S. N. Deepa, “Review on methods to fix number of hidden neurons in
   neural networks”, *Mathematical Problems in Engineering*, vol. 2013, art. 425740,
   2013. doi:10.1155/2013/425740
6. D. Stathakis, “How many hidden layers and nodes?”, *International Journal of Remote
   Sensing*, vol. 30, n.º 8, pp. 2133–2147, 2009. doi:10.1080/01431160802549278

## 4. Detector de ataques con UNSW-NB15

### Preparación de datos UNSW-NB15

*Responsable: P3 (T3).*

### 4.1 Clasificador UNSW-NB15 y normalización L2

*Responsable: P1 (T4).*

### 4.2 Ensamble y cuantificación de incertidumbre

*Responsable: P3 (T5).*

### 4.3 Expected calibration error

*Responsable: P2 (T6).*

#### Definición y decisiones adoptadas

**Definición del material de apoyo** (*Estimación de incertidumbre para modelos de
aprendizaje automático*, §4, en `docs/assignment/EstimacionIncertidumbre.pdf`). Dado un
estimador de incertidumbre $s(\vec{x}_i)$ y un conjunto de prueba con $N$ observaciones
etiquetadas, los puntajes se reparten en cubetas $B_1, \dots, B_{n_b}$. Para cada cubeta,
$\overline{\mathrm{acc}}(B_m)$ es la tasa de aciertos y $\overline{s}(B_m)$ el puntaje
promedio de sus observaciones:

$$\mathrm{ECE} = \sum_{m=1}^{n_b} \frac{|B_m|}{N} \left| \overline{\mathrm{acc}}(B_m) - \overline{s}(B_m) \right|$$

Las cubetas vacías se ignoran. El material supone puntajes en $[0, 1]$; si no lo están,
propone medir la correlación de Pearson entre incertidumbre y tasa de aciertos.

**Decisiones (D6 en `JOURNAL.md`).**

1. **Puntaje.** La incertidumbre del ensamble (§4.2) es una varianza: no está acotada y
   crece cuando el modelo es *menos* confiable, mientras que $\overline{\mathrm{acc}}$ mide
   aciertos. Por eso se usa la confianza $s = 1 - u_{\text{norm}}$, con $u_{\text{norm}}$ la
   incertidumbre normalizada min-max al rango del conjunto evaluado (o a un rango dado).
2. **Complemento.** Se reporta además la correlación de Pearson entre incertidumbre y
   acierto (1 o 0), la alternativa que ofrece el material. Un buen estimador da un valor
   negativo: más incertidumbre, menos aciertos.
3. **Cubetas.** 10 cubetas de igual ancho en $[0, 1]$ por defecto, como en la figura del
   material, configurables con `n_bins`. La cubeta $m$ cubre $[m/n_b, (m+1)/n_b)$; la
   última incluye el 1.
4. **Firma y relación con la última capa.** La tasa de aciertos necesita las predicciones,
   que la firma `calculate_expected_calibration_error(X, y, uncertainties)` no recibe. Se
   agrega el parámetro `predictions`: las etiquetas que el ensamble predice a partir de la
   salida de su última capa **antes** de la activación (los puntajes del producto punto),
   como pide el enunciado. Ese cálculo pertenece a §4.2; esta función recibe las etiquetas
   ya calculadas. `X` se conserva por la firma pedida y solo se usa para verificar que el
   número de observaciones coincida.

In [ ]:
import numpy as np


def uncertainty_to_confidence(uncertainties, u_range=None):
    """Convierte incertidumbres no negativas en puntajes de confianza en [0, 1].

    Calcula s = 1 - u_norm, donde u_norm es la incertidumbre normalizada
    min-max. La menor incertidumbre recibe confianza 1 y la mayor, 0.

    Parámetros
    ----------
    uncertainties : array_like, forma (n,)
        Incertidumbre por observación (por ejemplo, la varianza del ensamble).
        Debe tener valores finitos y no negativos.
    u_range : tupla (u_min, u_max) o None
        Rango de normalización. Si es None, se usan el mínimo y el máximo de
        `uncertainties`. Los valores fuera del rango se recortan a [0, 1].

    Retorna
    -------
    numpy.ndarray, forma (n,)
        Confianzas en [0, 1]. Si el rango es degenerado (u_max == u_min),
        todas valen 1, porque no hay diferencia de incertidumbre que ordenar.
    """
    u = np.asarray(uncertainties, dtype=float).ravel()
    if u.size == 0:
        raise ValueError("uncertainties no puede estar vacío")
    if not np.all(np.isfinite(u)) or np.any(u < 0):
        raise ValueError("uncertainties debe tener valores finitos y no negativos")
    if u_range is None:
        u_min, u_max = float(u.min()), float(u.max())
    else:
        u_min, u_max = (float(v) for v in u_range)
        if u_max < u_min:
            raise ValueError("u_range debe cumplir u_min <= u_max")
    if u_max == u_min:
        return np.ones_like(u)
    u_norm = np.clip((u - u_min) / (u_max - u_min), 0.0, 1.0)
    return 1.0 - u_norm


def calibration_bins(confidences, correct, n_bins=10):
    """Agrupa las observaciones en cubetas de igual ancho sobre [0, 1].

    La cubeta m cubre [m / n_bins, (m + 1) / n_bins); la última incluye 1.

    Parámetros
    ----------
    confidences : array_like, forma (n,)
        Puntajes de confianza en [0, 1].
    correct : array_like de bool o 0/1, forma (n,)
        Indica si la predicción de cada observación fue correcta.
    n_bins : int
        Número de cubetas (al menos 1).

    Retorna
    -------
    dict con
        'edges' : bordes de las cubetas, forma (n_bins + 1,).
        'count' : |B_m|, observaciones por cubeta, forma (n_bins,).
        'accuracy' : tasa de aciertos por cubeta; NaN si está vacía.
        'confidence' : confianza promedio por cubeta; NaN si está vacía.
    """
    c = np.asarray(confidences, dtype=float).ravel()
    ok = np.asarray(correct).ravel().astype(float)
    if c.size == 0:
        raise ValueError("confidences no puede estar vacío")
    if c.shape != ok.shape:
        raise ValueError("confidences y correct deben tener la misma longitud")
    if not np.all(np.isfinite(c)) or np.any((c < 0) | (c > 1)):
        raise ValueError("confidences debe estar en [0, 1]")
    if not np.all((ok == 0) | (ok == 1)):
        raise ValueError("correct debe contener solo valores booleanos o 0/1")
    if int(n_bins) != n_bins or n_bins < 1:
        raise ValueError("n_bins debe ser un entero positivo")
    n_bins = int(n_bins)

    idx = np.minimum(np.floor(c * n_bins).astype(int), n_bins - 1)
    count = np.bincount(idx, minlength=n_bins)
    hits = np.bincount(idx, weights=ok, minlength=n_bins)
    conf_sum = np.bincount(idx, weights=c, minlength=n_bins)
    filled = count > 0
    accuracy = np.full(n_bins, np.nan)
    confidence = np.full(n_bins, np.nan)
    accuracy[filled] = hits[filled] / count[filled]
    confidence[filled] = conf_sum[filled] / count[filled]
    return {
        "edges": np.linspace(0.0, 1.0, n_bins + 1),
        "count": count,
        "accuracy": accuracy,
        "confidence": confidence,
    }


def calculate_expected_calibration_error(X, y, uncertainties, predictions,
                                         n_bins=10, u_range=None):
    """Calcula el expected calibration error (ECE) de un estimador de incertidumbre.

    Sigue la definición del material de apoyo:
    ECE = sum_m |B_m| / N * |acc(B_m) - s(B_m)|, ignorando las cubetas vacías,
    con s = 1 - incertidumbre normalizada (ver `uncertainty_to_confidence`).

    Parámetros
    ----------
    X : array_like, forma (n, D)
        Observaciones evaluadas. Se conserva por la firma del enunciado y solo
        se usa para verificar que haya n observaciones.
    y : array_like, forma (n,)
        Etiquetas correctas (ground truth).
    uncertainties : array_like, forma (n,)
        Incertidumbre de cada observación (por ejemplo, varianza del ensamble).
    predictions : array_like, forma (n,)
        Etiquetas predichas a partir de la salida de la última capa antes de
        la activación (contrato con §4.2).
    n_bins : int
        Número de cubetas de igual ancho en [0, 1].
    u_range : tupla (u_min, u_max) o None
        Rango para normalizar la incertidumbre; None usa el del conjunto.

    Retorna
    -------
    float
        ECE en [0, 1]; 0 indica calibración perfecta.
    """
    X = np.asarray(X)
    if X.ndim == 0:
        raise ValueError("X debe tener al menos una dimensión")
    n = X.shape[0]
    y = np.asarray(y).ravel()
    predictions = np.asarray(predictions).ravel()
    confidences = uncertainty_to_confidence(uncertainties, u_range)
    if not (len(y) == len(predictions) == len(confidences) == n):
        raise ValueError("X, y, uncertainties y predictions deben tener n observaciones")
    bins = calibration_bins(confidences, predictions == y, n_bins)
    filled = bins["count"] > 0
    weights = bins["count"][filled] / n
    gaps = np.abs(bins["accuracy"][filled] - bins["confidence"][filled])
    return float(np.sum(weights * gaps))


def uncertainty_accuracy_correlation(uncertainties, y, predictions):
    """Correlación de Pearson entre la incertidumbre y el acierto (1 o 0).

    Es la alternativa del material de apoyo para puntajes fuera de [0, 1].
    Un estimador útil da un valor negativo: más incertidumbre, menos aciertos.

    Parámetros
    ----------
    uncertainties : array_like, forma (n,)
    y : array_like, forma (n,)
        Etiquetas correctas.
    predictions : array_like, forma (n,)
        Etiquetas predichas.

    Retorna
    -------
    float
        Coeficiente en [-1, 1], o NaN si la incertidumbre o los aciertos son
        constantes (la correlación no está definida).
    """
    u = np.asarray(uncertainties, dtype=float).ravel()
    correct = (np.asarray(predictions).ravel() == np.asarray(y).ravel()).astype(float)
    if u.shape != correct.shape or u.size < 2:
        raise ValueError("se necesitan al menos dos observaciones de igual longitud")
    if np.std(u) == 0 or np.std(correct) == 0:
        return float("nan")
    return float(np.corrcoef(u, correct)[0, 1])

#### Pruebas unitarias de §4.3

Las pruebas usan datos construidos a mano, con resultados que se pueden calcular sin la
función, y no dependen de otras secciones:

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `uncertainty_to_confidence` | Incertidumbres $[0, 2, 1]$ | Confianzas $[1, 0, 0.5]$ |
| | Rango fijo $(0, 1)$ con $[0.25, 1.5]$ | $[0.75, 0]$ (se recorta) |
| | Incertidumbre constante | Todas las confianzas valen 1 |
| | Incertidumbre negativa | `ValueError` |
| `calibration_bins` | Confianzas $[0.05, 0.15, 0.15, 1]$, aciertos $[1, 0, 1, 1]$ | Conteos 1, 2 y 1 en las cubetas 0, 1 y 9; tasa 0.5 en la cubeta 1 |
| | Confianza 1.2 | `ValueError` |
| `calculate_expected_calibration_error` | 4 obs. con confianza 0.75 y 3 aciertos; 4 con 0.25 y 1 acierto | ECE = 0 |
| | 10 obs. con confianza 0.95 y 8 aciertos; 10 con 0.55 y 5 aciertos | ECE = 0.5·0.15 + 0.5·0.05 = 0.10 |
| | Confianza 1 y todas incorrectas | ECE = 1 |
| | Incertidumbres $[0, 10]$ sin rango, primera correcta | ECE = 0 (normaliza y orienta) |
| | Longitudes distintas | `ValueError` |
| `uncertainty_accuracy_correlation` | Las dos incertidumbres altas fallan | $r = -1$ |
| | Todas correctas | NaN |

In [ ]:
import sys
import unittest


class TestUncertaintyToConfidence(unittest.TestCase):
    def test_extremos_y_orden(self):
        np.testing.assert_allclose(uncertainty_to_confidence([0.0, 2.0, 1.0]), [1.0, 0.0, 0.5])

    def test_rango_fijo_y_recorte(self):
        np.testing.assert_allclose(
            uncertainty_to_confidence([0.25, 1.5], u_range=(0.0, 1.0)), [0.75, 0.0])

    def test_incertidumbre_constante(self):
        np.testing.assert_allclose(uncertainty_to_confidence([3.0, 3.0, 3.0]), [1.0, 1.0, 1.0])

    def test_rechaza_negativos(self):
        with self.assertRaises(ValueError):
            uncertainty_to_confidence([0.1, -0.2])


class TestCalibrationBins(unittest.TestCase):
    def test_conteos_y_promedios(self):
        bins = calibration_bins([0.05, 0.15, 0.15, 1.0], [1, 0, 1, 1], n_bins=10)
        self.assertEqual(bins["count"].sum(), 4)
        self.assertEqual((bins["count"][0], bins["count"][1], bins["count"][9]), (1, 2, 1))
        self.assertAlmostEqual(bins["accuracy"][1], 0.5)
        self.assertAlmostEqual(bins["confidence"][1], 0.15)
        self.assertTrue(np.isnan(bins["accuracy"][5]))

    def test_rechaza_confianza_fuera_de_rango(self):
        with self.assertRaises(ValueError):
            calibration_bins([0.5, 1.2], [1, 0])


class TestExpectedCalibrationError(unittest.TestCase):
    @staticmethod
    def _caso(confianzas, aciertos):
        """Arma X, y, incertidumbres y predicciones con rango fijo (0, 1)."""
        n = len(confianzas)
        y = np.zeros(n, dtype=int)
        predictions = np.where(np.asarray(aciertos, dtype=bool), 0, 1)
        uncertainties = 1.0 - np.asarray(confianzas, dtype=float)
        return np.zeros((n, 2)), y, uncertainties, predictions

    def test_calibracion_perfecta(self):
        X, y, u, p = self._caso([0.75] * 4 + [0.25] * 4, [1, 1, 1, 0] + [1, 0, 0, 0])
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 0.0)

    def test_ejemplo_calculado_a_mano(self):
        X, y, u, p = self._caso([0.95] * 10 + [0.55] * 10, [1] * 8 + [0] * 2 + [1] * 5 + [0] * 5)
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 0.10)

    def test_peor_caso(self):
        X, y, u, p = self._caso([1.0] * 5, [0] * 5)
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 1.0)

    def test_normaliza_y_orienta_la_incertidumbre(self):
        ece = calculate_expected_calibration_error(
            np.zeros((2, 2)), y=[0, 0], uncertainties=[0.0, 10.0], predictions=[0, 1])
        self.assertAlmostEqual(ece, 0.0)

    def test_rechaza_longitudes_distintas(self):
        with self.assertRaises(ValueError):
            calculate_expected_calibration_error(np.zeros((3, 2)), [0, 1, 0], [0.1, 0.2], [0, 1, 0])


class TestUncertaintyAccuracyCorrelation(unittest.TestCase):
    def test_correlacion_negativa_perfecta(self):
        r = uncertainty_accuracy_correlation([0.0, 0.0, 1.0, 1.0], [0, 0, 0, 0], [0, 0, 1, 1])
        self.assertAlmostEqual(r, -1.0)

    def test_aciertos_constantes(self):
        r = uncertainty_accuracy_correlation([0.1, 0.5, 0.9], [1, 0, 1], [1, 0, 1])
        self.assertTrue(np.isnan(r))


suite_s4_3 = unittest.TestSuite()
for caso in (TestUncertaintyToConfidence, TestCalibrationBins,
             TestExpectedCalibrationError, TestUncertaintyAccuracyCorrelation):
    suite_s4_3.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_3 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_3)
assert resultado_s4_3.wasSuccessful(), "Fallaron pruebas unitarias de §4.3"

### 4.4 Evaluación de ensambles: ECE y tiempo de ejecución

*Responsable: P3 (T7).*